# Train an RVC v2 voice model on Google Colab
# Обучение RVC v2 в Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ltillegal/rvc-macos-sequoia/blob/main/colab/train-rvc-colab.ipynb)

This notebook trains an **RVC v2, 48 kHz, F0-conditioned** voice model on a free Colab GPU and puts the finished `.pth` + `.index` files back on your Google Drive. The result installs into [RVC-WebUI-MacOS](https://github.com/ltillegal/rvc-macos-sequoia) with the built-in ZIP installer.

Этот ноутбук обучает голосовую модель **RVC v2, 48 кГц, с F0** на бесплатном GPU в Colab и складывает готовые `.pth` + `.index` обратно на Google Drive. Файлы ставятся в RVC-WebUI-MacOS через встроенный ZIP-установщик.

> **Важно / Important:** RVC and Seed-VC are different architectures. An RVC `.pth` does **not** work in Seed-VC, and a Seed-VC checkpoint does not work here.
>
> RVC и Seed-VC — разные архитектуры. RVC-модель **не** открывается в Seed-VC, и наоборот.

**Runtime → Change runtime type → T4 GPU** before running. / Перед запуском включите GPU: *Runtime → Change runtime type → T4 GPU*.

> **If Colab disconnects / Если Colab отключился:** the runtime is gone but nothing important is lost. Click *Reconnect*, then **Runtime → Run all** - the notebook installs everything again (5-10 min), redoes preprocessing (1-3 min) and continues training from the checkpoints mirrored to your Drive.
>
> Рантайм пропадает, но ничего важного не теряется. Нажмите *Reconnect*, затем **Runtime → Run all**: ноутбук заново поставит зависимости (5-10 мин), повторит подготовку (1-3 мин) и продолжит обучение с чекпоинтов, скопированных на Drive.


## Before you start / Перед началом

**Dataset / Датасет**

- 10–30 minutes of one speaker is the sweet spot. 4 minutes works but the timbre will be less stable.
- Clean speech: no music, no echo, no second voice, no clipping, even level.
- WAV / MP3 / FLAC / M4A / OGG all work. Files are resampled automatically.
- Do not over-denoise: aggressive cleanup becomes part of the model ("metallic" voice).
- 10–30 минут одного голоса — оптимум. 4 минуты тоже обучатся, но тембр будет менее стабильным. Без музыки, эха, второго голоса и клиппинга. Агрессивный денойз вредит.

**Drive layout / Структура на Drive**

```
MyDrive/rvc/mymodel/dataset/      <- put your audio files here / сюда аудио
MyDrive/rvc/mymodel/out/          <- results appear here / сюда придут результаты
```

`mymodel` is just the default name — rename it freely (Latin letters, no spaces), the notebook uses it for the experiment folder and the output `.pth`.
`mymodel` — просто имя по умолчанию, можно поменять на любое (латиница, без пробелов): оно станет именем эксперимента и файлов `.pth`.

**Time / Время** — ~10–20 min preprocessing + ~40–90 min for 200 epochs on a free T4.

**Session limits / Лимиты сессии** — free Colab disconnects after a few hours of idle time. Keep the tab open. If it disconnects, run all cells again: preprocessing is cached locally, and training resumes from the checkpoints mirrored to Drive.


In [ ]:
#@title 1. Settings / Настройки { display-mode: "form" }
#@markdown Change the paths if your Drive layout is different. / Поменяйте пути, если структура другая.

EXPERIMENT   = "mymodel"  #@param {type:"string"}
DATASET_DIR  = "/content/drive/MyDrive/rvc/mymodel/dataset"  #@param {type:"string"}
OUTPUT_DIR   = "/content/drive/MyDrive/rvc/mymodel/out"  #@param {type:"string"}
SAMPLE_RATE  = "48k"  #@param ["48k", "40k", "32k"]
TOTAL_EPOCHS = 200  #@param {type:"integer"}
SAVE_EVERY   = 25  #@param {type:"integer"}
BATCH_SIZE   = 8  #@param {type:"integer"}
SPEAKER_ID   = 0  #@param {type:"integer"}
REPO_URL     = "https://github.com/ltillegal/rvc-macos-sequoia.git"  #@param {type:"string"}

import os

REPO_DIR    = "/content/rvc/repo"
VENV_DIR    = "/content/rvc/.venv"
PY          = os.path.join(VENV_DIR, "bin", "python")
LOCAL_DATA  = "/content/rvc/dataset"
TRAIN_DATA  = "/content/rvc/dataset_48k"
LOG_DIR     = os.path.join(REPO_DIR, "logs", EXPERIMENT)
WEIGHTS_DIR = os.path.join(REPO_DIR, "assets", "weights")
CKPT_BACKUP = os.path.join(OUTPUT_DIR, "checkpoints")

SR      = {"32k": 32000, "40k": 40000, "48k": 48000}[SAMPLE_RATE]
VERSION = "v2" if SAMPLE_RATE == "48k" else "v1"

if VERSION == "v2":
    PRETRAIN_G = os.path.join(REPO_DIR, "assets", "pretrained_v2", "f0G48k.pth")
    PRETRAIN_D = os.path.join(REPO_DIR, "assets", "pretrained_v2", "f0D48k.pth")
else:
    PRETRAIN_G = os.path.join(REPO_DIR, "assets", "pretrained", f"f0G{SAMPLE_RATE}.pth")
    PRETRAIN_D = os.path.join(REPO_DIR, "assets", "pretrained", f"f0D{SAMPLE_RATE}.pth")

print(f"Experiment : {EXPERIMENT}")
print(f"Sample rate: {SAMPLE_RATE} (RVC {VERSION}) / Частота: {SAMPLE_RATE}")
print(f"Epochs     : {TOTAL_EPOCHS} (checkpoint every {SAVE_EVERY})")
print(f"Dataset    : {DATASET_DIR}")
print(f"Output     : {OUTPUT_DIR}")

In [ ]:
#@title 2. Check the GPU / Проверка GPU
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU FOUND")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
#@title 3. Mount Google Drive / Подключить Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title 4. Install dependencies / Установка зависимостей (5-10 min, once per session)
import os, subprocess

def sh(cmd, **kw):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, check=True, **kw)

subprocess.run(["pip", "install", "-q", "uv"], check=True)
os.makedirs("/content/rvc", exist_ok=True)

if not os.path.exists(PY):
    sh(["uv", "venv", "--python", "3.11", VENV_DIR])

def uv(*args):
    sh(["uv", "pip", "install", "--python", PY, *args])

# torch 2.4.1 ships CUDA 12.1 wheels on PyPI and is known-good with this fairseq fork.
uv("torch==2.4.1", "torchaudio==2.4.1")
uv(
    "numpy==1.26.4", "scipy", "librosa==0.10.2", "soundfile", "numba", "llvmlite",
    "faiss-cpu", "praat-parselmouth", "scikit-learn",
    "tensorboard", "tensorboardX", "tqdm", "matplotlib", "PyYAML", "json5",
    "Cython", "ninja", "wheel", "setuptools==79.0.1", "packaging",
    "av", "ffmpeg-python", "omegaconf", "hydra-core>=1.3.2", "sacrebleu>=1.4.12",
)
# pyworld has no wheels for every Python build -> compile it against the venv numpy.
uv("--no-build-isolation", "pyworld")

# fairseq: clone the fork and build it against the installed torch.
# setup.py hardcodes -std=c++11 on Linux, which modern torch headers reject -> patch to C++17.
FAIRSEQ_DIR = "/content/rvc/fairseq"
if not os.path.isdir(FAIRSEQ_DIR):
    sh(["git", "clone", "--depth", "1", "https://github.com/One-sixth/fairseq.git", FAIRSEQ_DIR])

setup_py = os.path.join(FAIRSEQ_DIR, "setup.py")
with open(setup_py) as f:
    setup_src = f.read()
if '"-std=c++11"' in setup_src:
    with open(setup_py, "w") as f:
        f.write(setup_src.replace('"-std=c++11"', '"-std=c++17"'))
    print("patched fairseq setup.py: -std=c++11 -> -std=c++17")

# Keep the build CPU/C++ only: a visible CUDA_HOME makes fairseq compile .cu files
# against Colab's toolkit, which often mismatches the pinned torch version.
build_env = {
    k: v for k, v in os.environ.items()
    if k not in ("CUDA_HOME", "CUDA_PATH", "TORCH_CUDA_ARCH_LIST")
}
sh(["uv", "pip", "install", "--python", PY, "--no-build-isolation", FAIRSEQ_DIR], env=build_env)

# Verify every import separately: one broken module must not hide the others.
MODULES = [
    "torch", "torchaudio", "fairseq", "faiss", "librosa",
    "pyworld", "parselmouth", "numba", "soundfile", "av", "pkg_resources",
]
failed = []
print("import check / проверка импортов:")
for mod in MODULES:
    r = subprocess.run(
        [PY, "-c", f"import {mod}; print(getattr({mod}, '__version__', 'ok'))"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f"  {mod:14s} OK   {r.stdout.strip()}")
    else:
        failed.append(mod)
        last = (r.stderr.strip().splitlines() or ["unknown error"])[-1]
        print(f"  {mod:14s} FAIL {last}")

if failed:
    print()
    print(">>> Fix the failed module(s) before continuing / исправьте перед продолжением:", ", ".join(failed))
    if "pkg_resources" in failed or "pyworld" in failed:
        print(f'    uv pip install --python {PY} "setuptools<80"')

In [ ]:
#@title 5. Get the code and the required weights / Код и веса
import os, subprocess

def sh(cmd, **kw):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, check=True, **kw)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/"
FILES = {
    "hubert_base.pt":  "assets/hubert/hubert_base.pt",
    "rmvpe.pt":        "assets/rmvpe/rmvpe.pt",
}
if VERSION == "v2":
    FILES["pretrained_v2/f0G48k.pth"] = "assets/pretrained_v2/f0G48k.pth"
    FILES["pretrained_v2/f0D48k.pth"] = "assets/pretrained_v2/f0D48k.pth"
else:
    FILES[f"pretrained/f0G{SAMPLE_RATE}.pth"] = f"assets/pretrained/f0G{SAMPLE_RATE}.pth"
    FILES[f"pretrained/f0D{SAMPLE_RATE}.pth"] = f"assets/pretrained/f0D{SAMPLE_RATE}.pth"

for remote, local in FILES.items():
    dst = os.path.join(REPO_DIR, local)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024:
        print("cached:", local)
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    sh(["curl", "-L", "--fail", "--retry", "3", "-o", dst, BASE + remote])

print("weights ready / веса готовы")

In [ ]:
#@title 6. Copy the dataset to fast local disk / Копирование датасета
# Drive is slow: a single 1 GB file can take minutes and a folder with many
# small files can look "frozen". This cell prints one line per file, so you can
# see it is alive, and it resumes from a partial copy.
# Drive медленный: один файл на 1 ГБ может копироваться минутами, а папка из
# множества мелких файлов выглядит как зависание. Ячейка печатает строку на
# каждый файл и продолжает с места обрыва.
import os, shutil, time

if not os.path.isdir(DATASET_DIR):
    raise SystemExit(
        f"Dataset folder not found: {DATASET_DIR}\n"
        "Put your audio there (WAV/MP3/FLAC/M4A/OGG). / Положите аудио в эту папку."
    )

EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff")
os.makedirs(LOCAL_DATA, exist_ok=True)

print(f"scanning {DATASET_DIR} ...", flush=True)
src_files = []
for root, _dirs, names in os.walk(DATASET_DIR):
    for name in sorted(names):
        if name.lower().endswith(EXTS):
            src_files.append(os.path.join(root, name))
print(f"found {len(src_files)} audio file(s) / найдено файлов: {len(src_files)}", flush=True)
if not src_files:
    raise SystemExit("No audio files found. / Аудиофайлы не найдены.")

started = time.time()
total = 0
for i, src in enumerate(src_files, 1):
    name = os.path.basename(src)
    dst = os.path.join(LOCAL_DATA, name)
    src_size = os.path.getsize(src)
    if os.path.exists(dst) and os.path.getsize(dst) == src_size:
        total += src_size
        print(f"[{i}/{len(src_files)}] skip {name} (already copied)", flush=True)
        continue
    t0 = time.time()
    shutil.copy2(src, dst)
    dt = max(time.time() - t0, 1e-6)
    total += src_size
    print(
        f"[{i}/{len(src_files)}] {name}: "
        f"{src_size / 1024 / 1024:.1f} MB in {dt:.1f}s "
        f"({src_size / 1024 / 1024 / dt:.1f} MB/s)",
        flush=True,
    )

print(
    f"{len(src_files)} file(s), {total / 1024 / 1024:.1f} MB copied in "
    f"{(time.time() - started) / 60:.1f} min / файлов: {len(src_files)}, "
    f"{total / 1024 / 1024:.1f} МБ",
    flush=True,
)

In [ ]:
#@title 6b. Normalize and split into 10-minute parts / Нормализация и нарезка
# Why: RVC decodes the whole file into RAM before slicing, so one multi-hour
# file can kill the Colab runtime (session crash = out of memory).
# Splitting with ffmpeg keeps memory flat and gives preprocessing a uniform
# mono 48 kHz input, whatever the source format was.
# Зачем: RVC декодирует файл целиком в память, поэтому многочасовой файл
# валит Colab (падение сессии = не хватило памяти). Нарезка через ffmpeg
# держит память ровно и приводит всё к одному формату: моно 48 кГц.
import glob, os, subprocess, time

SEGMENT_SECONDS = 600  # 10 min per part / по 10 минут
DONE_DIR = "/content/rvc/.split_done"
os.makedirs(TRAIN_DATA, exist_ok=True)
os.makedirs(DONE_DIR, exist_ok=True)

EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff")
sources = [
    os.path.join(LOCAL_DATA, name)
    for name in sorted(os.listdir(LOCAL_DATA))
    if name.lower().endswith(EXTS)
]
if not sources:
    raise SystemExit(f"No audio in {LOCAL_DATA} / нет аудио в {LOCAL_DATA}")

started = time.time()
for i, src in enumerate(sources, 1):
    base = os.path.splitext(os.path.basename(src))[0]
    marker = os.path.join(DONE_DIR, base + ".done")
    if os.path.exists(marker):
        print(f"[{i}/{len(sources)}] skip {base} (already split)", flush=True)
        continue
    t0 = time.time()
    pattern = os.path.join(TRAIN_DATA, base + "_%03d.wav")
    cmd = [
        "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
        "-i", src,
        "-ac", "1", "-ar", str(SR), "-c:a", "pcm_s16le",
        "-f", "segment", "-segment_time", str(SEGMENT_SECONDS),
        "-reset_timestamps", "1",
        pattern,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:], flush=True)
        raise SystemExit(f"ffmpeg failed on {src} / ffmpeg не смог обработать {src}")
    open(marker, "w").close()
    parts = len(glob.glob(os.path.join(TRAIN_DATA, base + "_*.wav")))
    print(f"[{i}/{len(sources)}] {base}: {parts} part(s) in {time.time() - t0:.1f}s", flush=True)

parts = sorted(glob.glob(os.path.join(TRAIN_DATA, "*.wav")))
size_mb = sum(os.path.getsize(p) for p in parts) / 1024 / 1024
print(
    f"{len(parts)} part(s), {size_mb:.1f} MB in {TRAIN_DATA} / "
    f"частей: {len(parts)}, {size_mb:.1f} МБ",
    flush=True,
)
print(f"split took {(time.time() - started) / 60:.1f} min", flush=True)

In [ ]:
#@title 7. Preprocess: slice, resample, normalize / Нарезка и нормализация
# One process, no fork on purpose: forking after numba/librosa are imported can
# deadlock in Colab. The heartbeat prints every 5 s with the CPU time of the
# worker, so you can tell "still working" from "really stuck".
# Один процесс без fork: после импорта numba/librosa fork в Colab умеет
# подвисать. Каждые 5 секунд печатается сердцебиение с потраченным CPU, так
# что видно, работает процесс или действительно встал.
import os, subprocess, threading, time

os.makedirs(LOG_DIR, exist_ok=True)
log_file = os.path.join(LOG_DIR, "preprocess.log")
chunks_dir = os.path.join(LOG_DIR, "0_gt_wavs")

N_PROC = 1
NOPARALLEL = "True"

n_input = len(os.listdir(TRAIN_DATA)) if os.path.isdir(TRAIN_DATA) else 0
print(f"input: {TRAIN_DATA} ({n_input} file(s))", flush=True)
if n_input == 0:
    raise SystemExit(
        "dataset_48k is empty - run cell 6b first / "
        "папка пуста, сначала выполните ячейку 6b"
    )

cmd = [
    PY, "infer/modules/train/preprocess.py",
    TRAIN_DATA, str(SR), str(N_PROC), LOG_DIR, NOPARALLEL, "3.7",
]
print("+", " ".join(cmd), flush=True)


def chunk_count():
    return len(os.listdir(chunks_dir)) if os.path.isdir(chunks_dir) else 0


def cpu_seconds(pid):
    """utime+stime of the worker in seconds, -1 if the process is gone."""
    try:
        with open(f"/proc/{pid}/stat") as f:
            parts = f.read().split()
        return (int(parts[13]) + int(parts[14])) / os.sysconf("SC_CLK_TCK")
    except Exception:
        return -1.0


state = {"chunks": chunk_count(), "since": time.time(), "cpu": 0.0, "killed": False}
stop_beat = threading.Event()


def monitor(proc):
    t0 = time.time()
    while not stop_beat.wait(5):
        n = chunk_count()
        if n != state["chunks"]:
            state["chunks"] = n
            state["since"] = time.time()
        cpu = cpu_seconds(proc.pid)
        gain = (cpu - state["cpu"]) if cpu >= 0 else 0.0
        state["cpu"] = max(cpu, 0.0)
        idle = time.time() - state["since"]
        print(
            f"  ... {n} chunks, cpu +{gain:.1f}s, {idle:.0f}s since last chunk, "
            f"{(time.time() - t0) / 60:.1f} min total",
            flush=True,
        )
        # A flat CPU counter means the worker is blocked, not busy.
        stalled = cpu >= 0 and gain < 0.1
        limit = 180 if n == 0 else 900
        if stalled and idle > limit and not state["killed"]:
            state["killed"] = True
            print(
                f"no progress and no CPU for {idle:.0f}s - killing preprocess / "
                f"нет прогресса {idle:.0f} c, останавливаю",
                flush=True,
            )
            proc.kill()


started = time.time()
proc = subprocess.Popen(
    cmd, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env=dict(os.environ, PYTHONUNBUFFERED="1"),
)
print(f"worker pid {proc.pid}", flush=True)
beater = threading.Thread(target=monitor, args=(proc,), daemon=True)
beater.start()

try:
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
finally:
    stop_beat.set()

if rc != 0 or state["killed"]:
    print(f"\npreprocess failed (exit code {rc}), last lines of {log_file}:", flush=True)
    try:
        with open(log_file, errors="replace") as f:
            print("".join(f.readlines()[-40:]))
    except FileNotFoundError:
        pass
    print(f"input folder: {TRAIN_DATA}", flush=True)
    print("files there:", len(os.listdir(TRAIN_DATA)) if os.path.isdir(TRAIN_DATA) else "MISSING", flush=True)
    raise SystemExit(rc or 1)

for d in ("0_gt_wavs", "1_16k_wavs"):
    p = os.path.join(LOG_DIR, d)
    print(d, len(os.listdir(p)) if os.path.isdir(p) else 0, "files")
print(f"preprocess took {(time.time() - started) / 60:.1f} min", flush=True)

In [ ]:
#@title 8. Extract F0 with RMVPE / Извлечение F0 (rmvpe)
import subprocess
cmd = [
    PY, "infer/modules/train/extract/extract_f0_rmvpe.py",
    "1", "0", "0", LOG_DIR, "True",
]
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO_DIR, check=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("F0 done / F0 готово")

In [ ]:
#@title 9. Extract HuBERT features / Признаки HuBERT
import subprocess
cmd = [
    PY, "infer/modules/train/extract_feature_print.py",
    "cuda", "1", "0", "0", LOG_DIR, VERSION, "True",
]
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO_DIR, check=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("features done / признаки готовы")

In [ ]:
#@title 10. Write config.json and filelist.txt / Конфиг и список файлов
import json, os

feat_name = "3_feature768" if VERSION == "v2" else "3_feature256"
gt_dir    = os.path.join(LOG_DIR, "0_gt_wavs")
feat_dir  = os.path.join(LOG_DIR, feat_name)
f0_dir    = os.path.join(LOG_DIR, "2a_f0")
f0nsf_dir = os.path.join(LOG_DIR, "2b-f0nsf")

config_src = os.path.join(
    REPO_DIR, "configs", "v2" if VERSION == "v2" else "v1", f"{SAMPLE_RATE}.json"
)
with open(config_src) as f:
    cfg = json.load(f)
with open(os.path.join(LOG_DIR, "config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

def stems(directory, suffixes):
    out = set()
    for name in os.listdir(directory):
        for suf in suffixes:
            if name.endswith(suf):
                out.add(name[: -len(suf)])
                break
    return out

names = (
    stems(gt_dir, [".wav"])
    & stems(feat_dir, [".npy"])
    & stems(f0_dir, [".wav.npy"])
    & stems(f0nsf_dir, [".wav.npy"])
)
names = sorted(names)
if len(names) < 10:
    raise SystemExit(
        f"Only {len(names)} usable chunks. Check the dataset and re-run cells 7-9.\n"
        f"Всего {len(names)} фрагментов — проверьте датасет и повторите шаги 7-9."
    )

lines = [
    f"{gt_dir}/{n}.wav|{feat_dir}/{n}.npy|{f0_dir}/{n}.wav.npy|{f0nsf_dir}/{n}.wav.npy|{SPEAKER_ID}"
    for n in names
]
with open(os.path.join(LOG_DIR, "filelist.txt"), "w") as f:
    f.write("\n".join(lines))

total_sec = len(names) * 3.7
print(f"{len(names)} chunks / фрагментов (~{total_sec/60:.1f} min of audio)")
print("config.json + filelist.txt written / записаны")

In [ ]:
#@title 11. Resume from Drive if a previous run exists / Продолжить с Drive
import glob, os, shutil

os.makedirs(CKPT_BACKUP, exist_ok=True)
have_local = bool(glob.glob(os.path.join(LOG_DIR, "G_*.pth")))
saved = sorted(glob.glob(os.path.join(CKPT_BACKUP, "*.pth")))

if have_local:
    print("local checkpoints found, keeping them / локальные чекпоинты на месте")
elif saved:
    for p in saved:
        shutil.copy2(p, LOG_DIR)
    print(f"restored {len(saved)} checkpoint file(s) from Drive / восстановлено {len(saved)}")
else:
    print("no checkpoints yet: training will start from the pretrained model / начнём с претрейна")

In [ ]:
#@title 12. Train / Обучение (40-90 min on T4)
import glob, os, subprocess, time

os.makedirs(CKPT_BACKUP, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Mirror checkpoints to Drive every 5 minutes so a disconnect is recoverable.
sync = subprocess.Popen(
    ["bash", "-c",
     f'while true; do '
     f'cp -f "{LOG_DIR}"/G_*.pth "{LOG_DIR}"/D_*.pth "{CKPT_BACKUP}"/ 2>/dev/null; '
     f'cp -f "{WEIGHTS_DIR}"/{EXPERIMENT}_e*.pth "{CKPT_BACKUP}"/ 2>/dev/null; '
     f'sleep 300; done']
)

cmd = [
    PY, "infer/modules/train/train.py",
    "-e", EXPERIMENT,
    "-sr", SAMPLE_RATE,
    "-f0", "1",
    "-bs", str(BATCH_SIZE),
    "-g", "0",
    "-te", str(TOTAL_EPOCHS),
    "-se", str(SAVE_EVERY),
    "-pg", PRETRAIN_G,
    "-pd", PRETRAIN_D,
    "-l", "1",       # keep only the latest G/D (the synced copy is the safety net)
    "-c", "0",       # no GPU data caching
    "-sw", "1",      # also write inference-ready <name>_eXXX_sYYYY.pth
    "-v", VERSION,
]
print("+", " ".join(cmd), flush=True)

started = time.time()
rc = -1
try:
    rc = subprocess.call(cmd, cwd=REPO_DIR, env=dict(os.environ, PYTHONUNBUFFERED="1"))
finally:
    sync.terminate()

# train.py exits with os._exit(2333333) when it finishes on purpose -> 149 in a shell.
if rc not in (0, 149, 2333333):
    raise SystemExit(f"Training failed with exit code {rc} / обучение упало с кодом {rc}")

print(f"training finished in {(time.time() - started) / 60:.1f} min / обучение завершено")
print("inference checkpoints:")
for p in sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}_e*.pth"))):
    print(" ", os.path.basename(p), f"{os.path.getsize(p)/1024/1024:.0f} MB")

In [ ]:
#@title 13. Build the faiss index / Построение индекса
import glob, os
import faiss
import numpy as np

feat_name = "3_feature768" if VERSION == "v2" else "3_feature256"
feat_dir = os.path.join(LOG_DIR, feat_name)

npys = [np.load(p) for p in sorted(glob.glob(os.path.join(feat_dir, "*.npy")))]
if not npys:
    raise SystemExit("No features found. / Признаки не найдены.")
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)
print("features:", big_npy.shape)

if big_npy.shape[0] > 2e5:
    from sklearn.cluster import MiniBatchKMeans
    centers = min(10000, big_npy.shape[0])
    print(f"kmeans -> {centers} centers (a few minutes) / кластеризация...")
    big_npy = MiniBatchKMeans(
        n_clusters=centers, batch_size=1024, compute_labels=False, init="random", verbose=0
    ).fit(big_npy).cluster_centers_
    print("after kmeans:", big_npy.shape)

n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
n_ivf = max(n_ivf, 1)
index = faiss.index_factory(768 if VERSION == "v2" else 256, f"IVF{n_ivf},Flat")
faiss.extract_index_ivf(index).nprobe = 1
index.train(big_npy.astype(np.float32))
index.add(big_npy.astype(np.float32))

index_path = os.path.join(LOG_DIR, f"added_IVF{n_ivf}_Flat_nprobe_1_{EXPERIMENT}_{VERSION}.index")
faiss.write_index(index, index_path)
print("index:", os.path.basename(index_path), f"{os.path.getsize(index_path)/1024/1024:.0f} MB")

In [ ]:
#@title 14. Export .pth + .index to Google Drive / Экспорт на Drive
import glob, os, shutil

os.makedirs(OUTPUT_DIR, exist_ok=True)
exported = []

models = sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}_e*.pth")))
if not models:
    # fall back to the final checkpoint written by savee()
    models = sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}*.pth")))
for p in models:
    shutil.copy2(p, OUTPUT_DIR)
    exported.append(os.path.basename(p))

for p in glob.glob(os.path.join(LOG_DIR, "added_*.index")):
    shutil.copy2(p, OUTPUT_DIR)
    exported.append(os.path.basename(p))

print(f"exported {len(exported)} file(s) to {OUTPUT_DIR} / файлов выгружено: {len(exported)}")
for name in exported:
    print(" ", name)
print()
print("Next / Далее: download the .pth + .index, put them into one ZIP, then use")
print("'Install model from ZIP' in RVC-WebUI-MacOS, or copy the .pth to assets/weights/")
print("and the .index to logs/.")

## After training / После обучения

**1. Download / Скачать.** Take the latest `.pth` from `out/` (the highest epoch you like) and the `.index` next to it.

**2. Install on the Mac / Установить на Mac.** In RVC-WebUI-MacOS, open the **Install model from ZIP (.pth + .index)** panel, drop in a ZIP with both files, click **Install**. Manual fallback: `.pth` → `assets/weights/`, `.index` → `logs/`.

**3. Inference settings for speech / Настройки для речи**

| Setting | Value |
| --- | --- |
| F0 method | `rmvpe` |
| Pitch | `0` (or ±12 to change gender / ±12 для смены пола) |
| Index rate | `0.5–0.75` |
| Protect | `0.33` |
| RMS mix | `0.25` |

**4. Listen critically / Слушать критически.** Test on speech that was **not** in the dataset. If it sounds muffled, use an earlier epoch. If it sounds unstable, train 50–100 more epochs.

Проверяйте на речи, которой **не было** в датасете. Глухо — возьмите более раннюю эпоху. Нестабильно — добавьте 50–100 эпох.

## Troubleshooting / Если что-то пошло не так

| Problem | Fix |
| --- | --- |
| `CUDA out of memory` | Lower `BATCH_SIZE` to 6 or 4 and re-run cell 12. |
| `pyworld` fails with `No module named 'pkg_resources'` | setuptools 82+ removed it. Run `uv pip install --python $PY "setuptools<80"` and re-run cell 4. |
| `fairseq` build fails | Re-run cell 4. It patches `-std=c++11` to C++17 and builds with `--no-build-isolation`; if it still fails, Runtime → Disconnect and delete runtime, then run again. |
| Colab disconnected | Reconnect, run cells 1–14 again. Preprocessing is redone, training resumes from `out/checkpoints`. |
| Session crashes ("Your session crashed after using all available RAM") during preprocessing | One very long file: RVC decodes it entirely into memory. Cell 6b splits everything into 10-minute parts first - run 6, then 6b, then 7. For a huge file you can lower `SEGMENT_SECONDS` to 300. |\n| Cell 6 seems to hang with no output | Google Drive is slow and `shutil.copytree` prints nothing. The cell now walks the folder and prints one line per file. A 1 GB file at ~5 MB/s takes ~3.5 min; that is normal. If the speed shows 0.0 MB/s for minutes, remount Drive (cell 3) and re-run cell 6 - a partial copy is picked up where it stopped. |\n| Cell 7 shows `0 chunks` for minutes | The heartbeat prints the worker CPU every 5 s. Rising CPU + no chunks means it is still slicing; a flat CPU means it is blocked, and the cell kills it after 3 minutes and prints the log tail - send that output. Once it works you can try `N_PROC = 4`, `NOPARALLEL = "False"` for speed. |
| Cell 7 seems to hang with no output | Python buffers output when stdout is a pipe. Check `logs/mymodel/preprocess.log` (it is flushed on every line), or re-run with `PYTHONUNBUFFERED=1` (the notebook now sets it). |
| `Only N usable chunks` | The dataset is too short or has too much silence. Add clean speech and re-run cells 6, 6b, then 7–10. |
| Model sounds metallic | Fewer epochs, cleaner source, no aggressive denoise. Also try `Index rate` 0.3–0.5 at inference. |
| Wrong language in the UI | Not related to training; see the main README. |

---

If this notebook helped, you can support the project / если ноутбук помог — можно поддержать проект: [PayPal](https://www.paypal.com/donate/?business=0eskimo0%40gmail.com&no_recurring=0&item_name=Support+RVC-WebUI-MacOS&currency_code=USD)
